# Titanic model export: logistic regression + leak-free group feature

Trains the final model from my notebook *Titanic: Leak-Free Group Feature Beat Tuned Trees* on all training rows and saves it with `joblib`, so it can be published as a Kaggle Model and reused.

If the published model is attached as an input, the last section loads it and checks that it reproduces the same predictions.

In [1]:
import glob, os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
SEED = 42

# Works on Kaggle (/kaggle/input) and locally (data/raw)
DATA = os.path.dirname((glob.glob("/kaggle/input/**/train.csv", recursive=True) + glob.glob("../data/raw/train.csv"))[0])
# The project calls the sex column "Gender"
train = pd.read_csv(os.path.join(DATA, "train.csv")).rename(columns={"Sex": "Gender"})
test = pd.read_csv(os.path.join(DATA, "test.csv")).rename(columns={"Sex": "Gender"})
X, y = train.drop(columns="Survived"), train["Survived"]
print(train.shape, test.shape, f"survival rate {y.mean():.3f}")

(891, 12) (418, 11) survival rate 0.384


## Feature steps (same code as the main notebook)

In [3]:
COMMON_TITLES = ["Mr", "Mrs", "Miss", "Master"]
TITLE_ALIASES = {"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"}
NUMERIC, BINARY = ["Age", "LogFare"], ["AgeMissing", "HasCabin"]
CATEGORICAL = ["Pclass", "Gender", "Title", "FamilyType", "Embarked"]


def extract_title(names):
    titles = names.str.extract(r",\s*([^\.]+)\.", expand=False).str.strip().replace(TITLE_ALIASES)
    return titles.where(titles.isin(COMMON_TITLES), "Rare")


def add_features(df):
    out = df.copy()
    out["Title"] = extract_title(out["Name"])
    out["AgeMissing"] = out["Age"].isna().astype(int)
    out["HasCabin"] = out["Cabin"].notna().astype(int)
    out["LogFare"] = np.log1p(out["Fare"])
    family_size = out["SibSp"] + out["Parch"] + 1
    out["FamilyType"] = pd.cut(family_size, bins=[0, 1, 4, 11], labels=["alone", "small", "large"]).astype(str)
    return out


class TitleAgeImputer(BaseEstimator, TransformerMixin):
    """Fill missing Age with the median age of the passenger's title, learned in fit."""
    def fit(self, X, y=None):
        self.medians_ = X.groupby("Title")["Age"].median()
        self.global_median_ = X["Age"].median()
        return self

    def transform(self, X):
        out = X.copy()
        out["Age"] = out["Age"].fillna(out["Title"].map(self.medians_)).fillna(self.global_median_)
        return out

In [4]:
def group_keys(X):
    surname = X["Name"].str.split(",").str[0].str.strip()
    family = surname + "_" + X["Fare"].round(2).astype(str)
    return X["Ticket"].astype(str).to_numpy(), family.to_numpy()


class GroupSurvivalEncoder(BaseEstimator, TransformerMixin):
    """Adds GroupSurvival (rate of other known members, 0.5 if none) and GroupKnown (0/1)."""
    def __init__(self, default=0.5):
        self.default = default

    def fit(self, X, y):
        self.tickets_, self.families_ = group_keys(X)
        self.labels_ = np.asarray(y, dtype=float)
        return self

    def transform(self, X):               # new passengers: all stored labels may be used
        return self._encode(X, leave_one_out=False)

    def fit_transform(self, X, y):        # training rows: drop each passenger's own label
        return self.fit(X, y)._encode(X, leave_one_out=True)

    def _encode(self, X, leave_one_out):
        tickets, families = group_keys(X)
        rates, known = np.full(len(X), self.default), np.zeros(len(X), dtype=int)
        for i, (ticket, family) in enumerate(zip(tickets, families)):
            members = (self.tickets_ == ticket) | (self.families_ == family)
            if leave_one_out:
                members[i] = False
            if members.any():
                rates[i], known[i] = self.labels_[members].mean(), 1
        out = X.copy()
        out["GroupSurvival"], out["GroupKnown"] = rates, known
        return out


def make_preprocessor(group_survival=False):
    numeric = NUMERIC + (["GroupSurvival"] if group_survival else [])
    binary = BINARY + (["GroupKnown"] if group_survival else [])
    columns = ColumnTransformer([
        ("numeric", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
        ("binary", "passthrough", binary),
        ("categorical", make_pipeline(SimpleImputer(strategy="most_frequent"),
                                      OneHotEncoder(handle_unknown="ignore")), CATEGORICAL),
    ], verbose_feature_names_out=False)
    steps = [("engineer", FunctionTransformer(add_features)), ("impute_age", TitleAgeImputer())]
    if group_survival:
        steps.append(("group_survival", GroupSurvivalEncoder()))
    return Pipeline(steps + [("columns", columns)])


def make_model(estimator, group_survival=False):
    return Pipeline([("prep", make_preprocessor(group_survival)), ("model", estimator)])

## Train on all rows and save

In [ ]:
import joblib, json, sklearn

model = make_model(LogisticRegression(max_iter=1000), group_survival=True).fit(X, y)
os.makedirs("model", exist_ok=True)
joblib.dump(model, "model/titanic_lr_group.joblib")
pred = model.predict(test)
meta = {"model": "LogisticRegression + GroupSurvivalEncoder pipeline", "cv_accuracy_5fold": 0.8406,
        "kaggle_public": 0.79425, "sklearn": sklearn.__version__, "features": "see make_preprocessor()"}
json.dump(meta, open("model/metadata.json", "w"), indent=2)
pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": pred}).to_csv("submission.csv", index=False)
print(meta, f"| predicted survival rate {pred.mean():.3f}")

## Reload the published Kaggle Model (when attached as input)

In [ ]:
found = glob.glob("/kaggle/input/**/titanic_lr_group.joblib", recursive=True)
if found:
    loaded = joblib.load(found[0])
    same = (loaded.predict(test) == pred).all()
    print("loaded model from Kaggle Models:", found[0], "| identical predictions:", bool(same))
else:
    print("no attached model found; skipping reload check")